In [4]:
%load_ext autoreload
%autoreload 2
import pandas as pd
from datetime import datetime

from vnpy.trader.constant import Interval,Exchange
from vnpy.trader.database import BaseDatabase, get_database

In [2]:
# 原始数据读取
df_raw = pd.read_csv("/Users/jinhongdou/PyCharmMiscProject/github/vnpy_quant/data/cffex_main_roll_v26/IF/2024-01_IF2401.CFFEX.csv")

In [6]:
# 数据清洗
# TODO 不要全信脚本，关键点要查
# 凡是要进入模型、回测、收益评估的字段，必须回头读它的生成逻辑，并用原始样本抽检。
import pandas as pd
from clean_if_tick_raw import clean_if_tick_data, CleanConfig

config = CleanConfig(
    multiplier=300.0,
    tick_size=0.2,
    mild_gap_seconds=5.0,
    medium_gap_seconds=10.0,
    severe_gap_seconds=30.0,
)
# 总体数据，正常数据，正常数据进一步去掉某些异常行(目前主要用df_regular)
df_all, df_regular, df_regular_strict, report = clean_if_tick_data(df_raw, config)

for k, v in report["regular_flag_counts"].items():
    print(f"{k}: {v}")
for k, v in report["regular_gap_level_counts"].items():
    print(f"{k}: {v}")

flag_bad_datetime: 0
flag_non_regular_session: 0
flag_negative_delta_volume: 0
flag_negative_delta_turnover: 0
flag_non_positive_last_price: 0
flag_non_positive_l1_price: 0
flag_non_positive_limit_price: 0
flag_non_positive_ohl_price: 0
flag_any_non_positive_price: 0
flag_bad_spread: 0
flag_spread_not_tick_multiple: 0
flag_last_price_out_of_limit: 0
flag_bad_high_low_last: 0
flag_large_gap: 67
flag_after_medium_or_severe_gap: 6
flag_duplicate_symbol_datetime: 0
flag_last_volume_mismatch: 259353
flag_core_bad_raw: 0
flag_core_bad_regular: 0
normal: 358805
mild_gap: 61
severe_gap: 4
medium_gap: 2


In [ ]:
df_regular.columns

In [24]:
# 行情快照字段
snapshot_cols = [
    "last_price",
    "open_interest",
    "bid_price_1", "bid_price_2", "bid_price_3", "bid_price_4", "bid_price_5",
    "ask_price_1", "ask_price_2", "ask_price_3", "ask_price_4", "ask_price_5",
    "bid_volume_1", "bid_volume_2", "bid_volume_3", "bid_volume_4", "bid_volume_5",
    "ask_volume_1", "ask_volume_2", "ask_volume_3", "ask_volume_4", "ask_volume_5",
]
# 日内累计字段
cumulative_cols = [
    "volume",
    "turnover",
    "high_price",
    "low_price",
]
# 参考字段 / 合约规则字段
reference_cols = [
    "pre_close",
    "limit_up",
    "limit_down",
    "open_price",
]

# 数据概述

## 0 一句话描述

- 在 datetime 这个时间点，行情源对当前 IF 合约市场做了一次快照。

截至这个时间点，当日累计成交量是 volume，当日累计成交额是 turnover。
当前单边未平仓合约数量是 open_interest，也就是多头总持仓 = 空头总持仓 = open_interest。

limit_up 是当日涨停价，limit_down 是当日跌停价。
open_price 是今日开盘价。
high_price 是截至当前 tick 为止的今日最高成交价。
low_price 是截至当前 tick 为止的今日最低成交价。
last_price 是截至当前 tick 为止的最新成交价。

pre_close 是上一交易日的参考价字段，在期货里可能不是昨日最后一笔成交价，而更可能是上一交易日收盘价/结算价一类的参考值，具体需要用 limit_up、limit_down 反推验证。

当前订单簿里，最高的未成交买入报价是 bid_price_1，这个价位上挂着 bid_volume_1 手买单。
当前订单簿里，最低的未成交卖出报价是 ask_price_1，这个价位上挂着 ask_volume_1 手卖单。 **


在 datetime 这个交易所行情时间点，IF 某合约的最新成交价、累计成交量、累计成交额、持仓量、当日高低开、涨跌停价，以及买卖五档盘口状态。


## 1.1 标识类字段
| 字段             | 含义    | 说明                                                 |
| -------------- | ----- | -------------------------------------------------- |
| `gateway_name` | 行情网关名 | 例如 CTP、SIM、BACKTESTING、某数据源名称。通常不是市场数据本身，而是数据来源标识。 |
| `extra`        | 扩展字段  | vn.py 对象中可放额外信息。很多时候为空或字典。一般不直接用于特征。               |
| `symbol`       | 合约代码  | 例如 `IF2406`、`IF2409`。注意 IF 是品种，具体交易的是某个月份合约。       |
| `exchange`     | 交易所   | IF 属于中国金融期货交易所，中金所。                                |
| `name`         | 合约名称  | 例如沪深300股指期货某合约。一般展示用。                              |

## 1.2 时间字段
| 字段          | 含义     | 说明                                           |
| ----------- | ------ | -------------------------------------------- |
| `datetime`  | 行情时间   | 通常是交易所或行情源给出的 tick 时间，是你做回测和特征构造时最应该使用的事件时间。 |
| `localtime` | 本地接收时间 | 通常是本机或程序收到行情的时间。实盘里可用于延迟分析；历史回测中一般不要拿它当市场时间。 |

## 成交与持仓类字段
| 字段              | 含义                   | 类型          |
| --------------- | -------------------- | ----------- |
| `volume`        | 当日累计成交量              | 累计字段        |
| `turnover`      | 当日累计成交额              | 累计字段        |
| `open_interest` | 当前持仓量                | 状态快照字段      |
| `last_price`    | 最新成交价                | 快照字段        |
| `last_volume`   | 当前 tick 新增成交量，或最近成交量 | 增量/派生字段，需验证 |
- 这里的last_volume在真实数据里似乎一直为0

## 日内价格状态字段
| 字段           | 含义                  | 类型     |
| ------------ | ------------------- | ------ |
| `open_price` | 当日开盘价               | 日内状态字段 |
| `high_price` | 当日最高价               | 累计状态字段 |
| `low_price`  | 当日最低价               | 累计状态字段 |
| `pre_close`  | 昨日收盘价/前结算参考价，取决于数据源 | 参考字段   |
| `limit_up`   | 当日涨停价               | 交易规则字段 |
| `limit_down` | 当日跌停价               | 交易规则字段 |

## 买卖五档字段
- bid_price_X: 买X价
- ask_price_X: 卖X价
- bid_volume_1: 买X挂单量
- ask_volume_X: 卖X挂单量

## 最小语义表
| 字段组  | 字段                                           | 类型           | 使用建议                  |
| ---- | -------------------------------------------- | ------------ | --------------------- |
| 标识   | `symbol`, `exchange`, `name`, `gateway_name` | 元数据          | 分组、过滤                 |
| 时间   | `datetime`                                   | 事件时间         | 回测主时间                 |
| 时间   | `localtime`                                  | 接收时间         | 延迟检查，不做市场特征           |
| 成交   | `last_price`                                 | 最新成交快照       | 可用，但注意滞后              |
| 成交   | `volume`                                     | 当日累计         | 用 diff 转增量            |
| 成交   | `turnover`                                   | 当日累计成交额      | 用 diff 转增量            |
| 成交   | `last_volume`                                | tick 增量或近似增量 | 需和 `volume.diff()` 验证 |
| 持仓   | `open_interest`                              | 当前状态         | 可看变化，但不能还原开平细节        |
| 日内状态 | `open_price`, `high_price`, `low_price`      | 日内路径状态       | 可用，但注意语义              |
| 限价   | `limit_up`, `limit_down`                     | 交易规则         | 用于异常检查                |
| 盘口价格 | `bid_price_1~5`, `ask_price_1~5`             | 订单簿快照        | 构造 mid/spread/micro   |
| 盘口数量 | `bid_volume_1~5`, `ask_volume_1~5`           | 订单簿深度        | 构造 depth/imbalance    |

## 不同时间尺度交易的区别
| 交易尺度    | 主要信息           | 核心难点            |
| ------- | -------------- | --------------- |
| tick/秒级 | 盘口、成交、短期冲击     | 噪声极大、成本敏感、执行要求高 |
| 分钟级日内   | 放量、趋势、波动、日内结构  | 假突破、追高杀跌、状态切换   |
| 日频/跨日   | 趋势、宏观、波动率、资金状态 | 隔夜跳空、慢变量失效      |
| 更长周期    | 基本面、政策、估值、配置   | 回撤周期长、反馈慢       |



In [9]:
# 数据列分区

# 最小信息量分区
minimal_cols = [
    "datetime",
    "last_price",
    "volume",
    "delta_volume",
    "turnover",
    "delta_turnover",
    "tick_vwap",
    "bid_price_1",
    "ask_price_1",
    "spread",
]

# 成交视图
trade_cols = [
    "datetime",
    "last_price",
    "volume",
    "delta_volume",
    "turnover",
    "delta_turnover",
    "tick_vwap",
    "cum_vwap",
]

# 盘口视图
book_cols = [
    "datetime",
    "bid_price_1",
    "bid_volume_1",
    "ask_price_1",
    "ask_volume_1",
    "mid_price",
    "spread",
    "obi_1",
    "micro_price",
]

# 持仓视图
oi_cols = [
    "datetime",
    "last_price",
    "delta_volume",
    "open_interest",
    "delta_open_interest",
]

# 数据质量视图
quality_cols = [
    "datetime",
    "session",
    "dt_gap_seconds",
    "gap_level",
    "flag_core_bad_regular",
    "flag_after_medium_or_severe_gap",
]

In [13]:
# 第一个练习
# 你能解释 volume 和 delta_volume 的区别；
    # 总交易数和前一个tick时间段的交易数
# 你能解释 turnover 和 delta_turnover 的区别；
# 你看到 delta_volume > 0 时，能理解 tick_vwap 是这段 tick 间隔的成交均价；
    # 与last_price的区别，last_price是上一个tick最后一次交易的价格
# 你能判断 bid_price_1 / ask_price_1 / spread 是否基本正常。
    # 当前切片的一档盘口正常：bid < ask，spread 为 tick_size 的整数倍，且主要在 1-2 tick。
    # 主动单负责触发成交，被动挂单决定成交价。
df_regular[minimal_cols].iloc[1000:1020]

,datetime,last_price,volume,delta_volume,turnover,delta_turnover,tick_vwap,bid_price_1,ask_price_1,spread
1003,2024-01-02 09:38:20.100,3417.8,7662,0.0,7.872866e+09,0.0,NaN,3417.4,3417.6,0.2
1004,2024-01-02 09:38:20.600,3417.4,7664,2.0,7.874916e+09,2050440.0,3417.400000,3417.2,3417.6,0.4
1005,2024-01-02 09:38:21.100,3417.2,7666,2.0,7.876966e+09,2050440.0,3417.400000,3417.2,3417.6,0.4
1006,2024-01-02 09:38:21.600,3417.6,7667,1.0,7.877992e+09,1025280.0,3417.600000,3417.2,3417.6,0.4
1007,2024-01-02 09:38:22.100,3417.6,7668,1.0,7.879017e+09,1025280.0,3417.600000,3417.2,3417.6,0.4
1008,2024-01-02 09:38:22.600,3417.6,7670,2.0,7.881067e+09,2050500.0,3417.500000,3417.4,3417.6,0.2
1009,2024-01-02 09:38:23.100,3417.6,7671,1.0,7.882093e+09,1025280.0,3417.600000,3417.4,3417.6,0.2
1010,2024-01-02 09:38:23.600,3417.6,7672,1.0,7.883118e+09,1025280.0,3417.600000,3417.4,3417.6,0.2
1011,2024-01-02 09:38:24.100,3417.6,7676,4.0,7.887219e+09,4101120.0,3417.600000,3417.4,3417.6,0.2
1012,2024-01-02 09:38:24.600,3417.6,7687,11.0,7.898497e+09,11277840.0,3417.527273,3417.4,3417.6,0.2


In [15]:
# 练习样本
print(df_regular[minimal_cols].iloc[1000:1020].to_string())

                    datetime  last_price  volume  delta_volume      turnover  delta_turnover    tick_vwap  bid_price_1  ask_price_1  spread
1003 2024-01-02 09:38:20.100      3417.8    7662           0.0  7.872866e+09             0.0          NaN       3417.4       3417.6     0.2
1004 2024-01-02 09:38:20.600      3417.4    7664           2.0  7.874916e+09       2050440.0  3417.400000       3417.2       3417.6     0.4
1005 2024-01-02 09:38:21.100      3417.2    7666           2.0  7.876966e+09       2050440.0  3417.400000       3417.2       3417.6     0.4
1006 2024-01-02 09:38:21.600      3417.6    7667           1.0  7.877992e+09       1025280.0  3417.600000       3417.2       3417.6     0.4
1007 2024-01-02 09:38:22.100      3417.6    7668           1.0  7.879017e+09       1025280.0  3417.600000       3417.2       3417.6     0.4
1008 2024-01-02 09:38:22.600      3417.6    7670           2.0  7.881067e+09       2050500.0  3417.500000       3417.4       3417.6     0.2
1009 2024-01-02 09:3

In [16]:
# 第二个练习
# bid_volume_1 / ask_volume_1 分别表示什么
    # last_price：已经发生的最新成交价
# mid_price = (bid_price_1 + ask_price_1) / 2，它和 last_price 有什么区别？
    # mid_price：当前盘口买卖报价的中点参考价 
# obi_1 是买一卖一挂单量的不平衡，正负分别代表什么？
    # Order Book Imbalance
    # 计算方法是 买一卖一挂单量之差 / 买一卖一挂单量之和
# micro_price 为什么会向挂单量较少的一侧偏移？
    # micro_price 是一个反向加权平均。数学上会向mid_price挂单更少的一方偏移(或者说向买盘或者卖盘更薄的方向偏移)
    # micro_price = 基于一档盘口不平衡修正后的 mid_price。 或者一个粗略的，瞬时的交易均价预期
# 注意考虑撤单的情况
    # 挂单量 ≠ 真实一定愿意成交的量，有些挂单是真承接，有些可能只是短暂报价、试探、诱导、做市、排队占位，下一秒就撤。
# 底层有人在拼延时，但你现在这份数据不是那个战场的完整地图。
# 关于撤单时延，不能把挂单当成“想成交就成交，不想成交就撤”的免费选择权。
book_cols = [
    "datetime",
    "last_price",
    "bid_price_1",
    "bid_volume_1",
    "ask_price_1",
    "ask_volume_1",
    "mid_price",
    "spread",
    "obi_1",
    "micro_price",
]

df_regular[book_cols].iloc[1000:1020]

,datetime,last_price,bid_price_1,bid_volume_1,ask_price_1,ask_volume_1,mid_price,spread,obi_1,micro_price
1003,2024-01-02 09:38:20.100,3417.8,3417.4,1,3417.6,3,3417.5,0.2,-0.500000,3417.450000
1004,2024-01-02 09:38:20.600,3417.4,3417.2,4,3417.6,4,3417.4,0.4,0.000000,3417.400000
1005,2024-01-02 09:38:21.100,3417.2,3417.2,3,3417.6,3,3417.4,0.4,0.000000,3417.400000
1006,2024-01-02 09:38:21.600,3417.6,3417.2,3,3417.6,2,3417.4,0.4,0.200000,3417.440000
1007,2024-01-02 09:38:22.100,3417.6,3417.2,3,3417.6,1,3417.4,0.4,0.500000,3417.500000
1008,2024-01-02 09:38:22.600,3417.6,3417.4,2,3417.6,1,3417.5,0.2,0.333333,3417.533333
1009,2024-01-02 09:38:23.100,3417.6,3417.4,2,3417.6,2,3417.5,0.2,0.000000,3417.500000
1010,2024-01-02 09:38:23.600,3417.6,3417.4,2,3417.6,1,3417.5,0.2,0.333333,3417.533333
1011,2024-01-02 09:38:24.100,3417.6,3417.4,2,3417.6,3,3417.5,0.2,-0.200000,3417.480000
1012,2024-01-02 09:38:24.600,3417.6,3417.4,1,3417.6,1,3417.5,0.2,0.000000,3417.500000


In [17]:
print(df_regular[book_cols].iloc[1000:1020].to_string())

                    datetime  last_price  bid_price_1  bid_volume_1  ask_price_1  ask_volume_1  mid_price  spread     obi_1  micro_price
1003 2024-01-02 09:38:20.100      3417.8       3417.4             1       3417.6             3     3417.5     0.2 -0.500000  3417.450000
1004 2024-01-02 09:38:20.600      3417.4       3417.2             4       3417.6             4     3417.4     0.4  0.000000  3417.400000
1005 2024-01-02 09:38:21.100      3417.2       3417.2             3       3417.6             3     3417.4     0.4  0.000000  3417.400000
1006 2024-01-02 09:38:21.600      3417.6       3417.2             3       3417.6             2     3417.4     0.4  0.200000  3417.440000
1007 2024-01-02 09:38:22.100      3417.6       3417.2             3       3417.6             1     3417.4     0.4  0.500000  3417.500000
1008 2024-01-02 09:38:22.600      3417.6       3417.4             2       3417.6             1     3417.5     0.2  0.333333  3417.533333
1009 2024-01-02 09:38:23.100      3417.6 

In [23]:
# phase 3 持仓视图
oi_cols = [
    "datetime",
    "last_price",
    "delta_volume",
    "open_interest",
    "delta_open_interest",
]

"""
1. open_interest 和 volume 的区别是什么？
  open_interest = 于当前市场上未平仓合约的单边数量，也就是多头总手数 = 空头总手数。
2. delta_open_interest > 0 代表什么？
  新开仓占据主导
3. delta_open_interest < 0 代表什么？
  2的反面
4. 为什么 delta_volume > 0 时，delta_open_interest 可能为正、负、或 0？

    delta_volume > 0：说明发生了成交。

    delta_open_interest > 0：说明净开仓增加，新合约生成占主导。
    delta_open_interest < 0：说明净平仓增加，旧合约消失占主导。
    delta_open_interest = 0：说明未平仓合约净数量没变，可能是一开一平的换手，也可能是开仓和平仓在这个 tick 区间内抵消。
"""

df_regular[oi_cols].iloc[1000:1020]

,datetime,last_price,delta_volume,open_interest,delta_open_interest
1003,2024-01-02 09:38:20.100,3417.8,0.0,102246,0.0
1004,2024-01-02 09:38:20.600,3417.4,2.0,102245,-1.0
1005,2024-01-02 09:38:21.100,3417.2,2.0,102246,1.0
1006,2024-01-02 09:38:21.600,3417.6,1.0,102247,1.0
1007,2024-01-02 09:38:22.100,3417.6,1.0,102247,0.0
1008,2024-01-02 09:38:22.600,3417.6,2.0,102247,0.0
1009,2024-01-02 09:38:23.100,3417.6,1.0,102246,-1.0
1010,2024-01-02 09:38:23.600,3417.6,1.0,102245,-1.0
1011,2024-01-02 09:38:24.100,3417.6,4.0,102242,-3.0
1012,2024-01-02 09:38:24.600,3417.6,11.0,102241,-1.0


In [24]:
# phase 4: 成交量 + 持仓量联合解读。
phase4_cols = [
    "datetime",
    "last_price",
    "delta_volume",
    "open_interest",
    "delta_open_interest",
    "bid_price_1",
    "ask_price_1",
    "spread",
]

# 真实片段
print(df_regular[phase4_cols].iloc[1000:1020].to_string())
# 合成片段
"""
                    datetime  last_price  delta_volume  open_interest  delta_open_interest  bid_price_1  ask_price_1  spread
2000 2024-01-02 10:15:00.000      3420.0           0.0         101000                  0.0       3419.8       3420.0     0.2
2001 2024-01-02 10:15:00.500      3419.8           8.0         101006                  6.0       3419.6       3419.8     0.2
2002 2024-01-02 10:15:01.000      3419.6          15.0         101018                 12.0       3419.4       3419.6     0.2
2003 2024-01-02 10:15:01.500      3419.4          22.0         101035                 17.0       3419.2       3419.4     0.2
2004 2024-01-02 10:15:02.000      3419.2          30.0         101060                 25.0       3419.0       3419.2     0.2
# 下跌 + 增仓 = 空头新开仓压力偏强

2005 2024-01-02 10:15:02.500      3419.2          18.0         101058                 -2.0       3419.0       3419.2     0.2
2006 2024-01-02 10:15:03.000      3419.4          25.0         101045                -13.0       3419.4       3419.6     0.2
2007 2024-01-02 10:15:03.500      3419.6          32.0         101020                -25.0       3419.6       3419.8     0.2
2008 2024-01-02 10:15:04.000      3419.8          28.0         100998                -22.0       3419.8       3420.0     0.2
# 上涨 + 减仓 = 空头平仓推动反弹

2009 2024-01-02 10:15:04.500      3419.8          12.0         100999                  1.0       3419.6       3420.0     0.4
2010 2024-01-02 10:15:05.000      3419.6          10.0         100999                  0.0       3419.4       3419.8     0.4
2011 2024-01-02 10:15:05.500      3419.8          14.0         101000                  1.0       3419.6       3420.0     0.4
2012 2024-01-02 10:15:06.000      3419.6          11.0         100999                 -1.0       3419.4       3419.8     0.4
# 震荡 + OI 基本不变 + 成交量回落 = 换手震荡 / 多空暂时均衡
"""

# 核心基本逻辑
# 涨 + 增仓：新多推动
# 涨 + 减仓：空头回补
# 跌 + 增仓：新空推动
# 跌 + 减仓：多头离场
# 震荡 + OI 基本不变 + 成交量回落 = 换手震荡 / 
    # 成交还在发生，但净持仓没有明显增加或减少；
    # 市场上的仓位更多是在不同玩家之间转移，而不是整体大量新增或整体大量消失。

# 期货交易本身：多空零和
# 期货价格来源：对未来现货指数的预期
# 外部冲击：改变大家对未来指数的预期
# 股指期货本来的作用: 方便做空，方便大资金的股市玩家做风险对冲

# 方向判断错误会亏钱；
# 仓位管理错误会出局。 -- 你可以猜对长期方向，但如果杠杆太高，中途波动就能先把你踢出局。

                    datetime  last_price  delta_volume  open_interest  delta_open_interest  bid_price_1  ask_price_1  spread
1003 2024-01-02 09:38:20.100      3417.8           0.0         102246                  0.0       3417.4       3417.6     0.2
1004 2024-01-02 09:38:20.600      3417.4           2.0         102245                 -1.0       3417.2       3417.6     0.4
1005 2024-01-02 09:38:21.100      3417.2           2.0         102246                  1.0       3417.2       3417.6     0.4
1006 2024-01-02 09:38:21.600      3417.6           1.0         102247                  1.0       3417.2       3417.6     0.4
1007 2024-01-02 09:38:22.100      3417.6           1.0         102247                  0.0       3417.2       3417.6     0.4
1008 2024-01-02 09:38:22.600      3417.6           2.0         102247                  0.0       3417.4       3417.6     0.2
1009 2024-01-02 09:38:23.100      3417.6           1.0         102246                 -1.0       3417.4       3417.6     0.2


In [35]:
# 盘口 + 成交联合判断主动买 / 主动卖。
# phase5_cols = [
    "datetime",
    "last_price",
    "tick_vwap",
    "delta_volume",
    "bid_price_1",
    "bid_volume_1",
    "ask_price_1",
    "ask_volume_1",
    "spread",
    "obi_1",
    "micro_price",
]
# 合成片段
"""                    
datetime  last_price  tick_vwap  delta_volume  bid_price_1  bid_volume_1  ask_price_1  ask_volume_1  spread  obi_1  micro_price
3000 2024-01-02 10:30:00.000      3420.0        NaN           0.0       3419.8            10       3420.0             8     0.2   0.11  3419.911
3001 2024-01-02 10:30:00.500      3420.0     3420.0          12.0       3419.8             9       3420.0             2     0.2   0.64  3419.964
3002 2024-01-02 10:30:01.000      3420.2     3420.1          18.0       3420.0             7       3420.2             5     0.2   0.17  3420.117
3003 2024-01-02 10:30:01.500      3420.2     3420.2          16.0       3420.0             8       3420.2             1     0.2   0.78  3420.178
3004 2024-01-02 10:30:02.000      3420.4     3420.3          24.0       3420.2             6       3420.4             4     0.2   0.20  3420.320

3005 2024-01-02 10:30:02.500      3420.4     3420.4          20.0       3420.2             4       3420.4            18     0.2  -0.64  3420.236
3006 2024-01-02 10:30:03.000      3420.2     3420.3          22.0       3420.2             3       3420.4            16     0.2  -0.68  3420.232
3007 2024-01-02 10:30:03.500      3420.2     3420.2          15.0       3420.0             8       3420.2            14     0.2  -0.27  3420.073
3008 2024-01-02 10:30:04.000      3420.0     3420.1          26.0       3420.0             2       3420.2            12     0.2  -0.71  3420.029
"""
# 1. 哪一段更像买方主动推进？
# 2. 哪一段更像卖方主动压制 / 主动卖更强？
# 3. 依据是什么？从 last_price、tick_vwap、bid/ask 挂单变化、obi_1、micro_price 里挑 2-3 个证据。


# bid_volume_1 > ask_volume_1
# obi_1 为正
# micro_price 靠近 ask
# 这几个信息大概能看出买一的接盘比较厚，但是未必说明主动买正在吃盘，因为挂单多不代表主动成交多

# 3001-3004真正的主动买的强证据
#     1. tick_vwap 多次靠近 ask，说明成交更偏主动买；(核心证据)
#     2. ask_volume_1 被吃薄后 ask_price_1 上移； 
#     3. last_price 和 bid/ask 报价整体抬升；
#     4. OBI 为正、micro_price 偏上，说明买一承接更厚，是辅助证据。

# 几个概念重新讨论
# last_price / tick_vwap：看成交发生在哪里
# bid/ask price：看盘口档位有没有推进
# bid/ask volume：看当前剩余队列厚度
# obi / micro_price：看当前盘口压力方向

# 盘厚 ≠ 主动卖。
    # 卖方盘厚 = 上方有人大量挂卖单，形成压制。
    # 主动卖 = 有人直接对着 bid 成交，把买盘打掉。

# 最小记忆
  # last_price 看刚刚成交在哪里；obi/micro_price 看当前盘口压力偏哪边；下一个 last_price 才会验证这个压力有没有兑现。如果下一个 last_price和上一次的micro_price近似相等，才等于压力兑现了
  # 一些常见情况
      # 主动买推进: 买方吃卖一，卖一被吃薄，ask 上移，价格抬升。
      # 反过来
      # 吸收/压制：有主动成交，但价格推不动，对手盘持续补单。

IndentationError: unexpected indent (161231421.py, line 3)